# Single-sample hallucination behavior analysis

This notebook uses `SampleBehaviorVisualizer` to inspect one test sample at a time. Hallucination spans come from `labels.jsonl`; graph features are computed directly from the reconstructed canonical attention CSR.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from behavior_analysis import BehaviorAnalysis
from research_analysis import SampleBehaviorVisualizer

DATA_ROOT = Path(
    '/share/home/tm902089733300000/a903202310/lys/data/RAGTruth/'
    'model_traces/llama31_8b/test'
)
OUTPUT_ROOT = Path(
    '/share/home/tm902089733300000/a903202310/lys/data/RAGTruth/'
    'visualizations/llama31_8b/token_behavior/test'
)

viewer = SampleBehaviorVisualizer(DATA_ROOT, device='cpu', verify_hashes=False)
print(f'total={len(viewer.dataset)}, correct={len(viewer.correct_sample_ids)}, error={len(viewer.error_sample_ids)}')


## Find an error sample

Run this cell first if you do not know a hallucinated `sample_id`.


In [ ]:
viewer.list_errors(limit=10)


## Inspect one sample

Replace `ERROR_SAMPLE_ID` with any ID returned above. The shaded region is the labeled hallucination span.


In [ ]:
ERROR_SAMPLE_ID = viewer.error_sample_ids[0]  # replace with a specific sample_id

record = viewer.analyze(ERROR_SAMPLE_ID)
print(record['metadata'])
print('positive_runs:', record['positive_runs'])
print('response_token_ids:', record['response_token_ids'].tolist())
viewer.plot(ERROR_SAMPLE_ID)


## Compare error vs pre-error within the same response

`span_summary` reports pre/error/post means and `error_minus_pre` for every structural feature.


In [ ]:
viewer.span_summary(ERROR_SAMPLE_ID, pre_window=8, post_window=8)


## Compare with a fully correct control

Calling `compare(ERROR_SAMPLE_ID)` automatically chooses a fully correct control, preferring the same source/task and then similar response length. You can also pass a second sample ID manually.


In [ ]:
figure, CORRECT_SAMPLE_ID = viewer.compare(ERROR_SAMPLE_ID)
print('matched correct sample:', CORRECT_SAMPLE_ID)

# Manual comparison example:
# viewer.compare(ERROR_SAMPLE_ID, '12345')


## Token-level t-SNE for this graph

Each point is one response token. The 11 graph-behavior features are standardized and embedded without labels; hallucination spans are loaded only afterwards for coloring. The path connects tokens in generation order, and the second panel colors the same coordinates by normalized response position.

In [ ]:
from IPython.display import Image, display

sample_output = OUTPUT_ROOT / ERROR_SAMPLE_ID
BehaviorAnalysis(DATA_ROOT, sample_output).single(ERROR_SAMPLE_ID)
display(Image(sample_output / 'token_tsne.png'))